# STAGE 1: Kaggle T4 Training - 30K Hard Subset

**UPDATED**: Paths match với cấu trúc dataset thực tế trên Kaggle

**Datasets cần add**:
- `aicity-30k-hard-enhanced` (data + manifest)
- `BBOX_dataset` (boxes_30k.json)
- `ckpt_30k_hard` (checkpoints)

**Goal**: mAP 80% → 82-86%

## 1. Setup Environment

In [ ]:
# Clone repo
!git clone https://github.com/Khanhhh239/Model_XVLM_Training.git
%cd Model_XVLM_Training/trainv4

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q albumentations torch==2.1.0 torchvision==0.16.0
!pip install -q -e .

## 2. Extract & Link Data

In [ ]:
import os
from pathlib import Path
import shutil

# Create data directories
Path("data").mkdir(exist_ok=True)
Path("data/checkpoints").mkdir(parents=True, exist_ok=True)

print("📦 Extracting train_30k_hard_data.tar.zst...")
print("   This may take 5-10 minutes...")

# Extract .tar.zst using tar (Kaggle has zstd support)
!tar -I zstd -xf /kaggle/input/aicity-30k-hard-enhanced/train_30k_hard_data.tar.zst -C data/

print("✓ Extraction completed!")

In [ ]:
# Link checkpoint files
print("🔗 Linking checkpoint...")
shutil.copy(
    "/kaggle/input/ckpt-30k-hard/best.pth",
    "data/checkpoints/best.pth"
)
print(f"✓ Checkpoint: {Path('data/checkpoints/best.pth').stat().st_size / (1024**2):.1f} MB")

# Link boxes JSON
print("\n🔗 Linking boxes...")
shutil.copy(
    "/kaggle/input/bbox-dataset/boxes_30k.json",
    "data/boxes_30k.json"
)
print(f"✓ Boxes: {Path('data/boxes_30k.json').stat().st_size / (1024**2):.1f} MB")

# Link manifest (if using parquet)
if Path("/kaggle/input/ckpt-30k-hard/manifest_30k_hard_enhanced.parquet").exists():
    shutil.copy(
        "/kaggle/input/ckpt-30k-hard/manifest_30k_hard_enhanced.parquet",
        "data/manifest_30k_hard_enhanced.parquet"
    )
    print("✓ Manifest (parquet)")

In [ ]:
# Verify extracted data structure
print("📁 Checking extracted files...\n")

# Expected structure after extraction:
# data/
#   train_30k_hard_data/   ← từ .tar.zst
#     ├── images/ hoặc train_webp/
#     ├── train_30k_hard.jsonl
#     └── train_30k_hard_vitpose.json

# Check what was extracted
extracted_dir = Path("data/train_30k_hard_data")
if extracted_dir.exists():
    print(f"✓ Extracted to: {extracted_dir}")
    for item in extracted_dir.iterdir():
        if item.is_dir():
            num_files = len(list(item.glob("*")))
            print(f"  📂 {item.name}/ ({num_files} files)")
        else:
            size_mb = item.stat().st_size / (1024**2)
            print(f"  📄 {item.name} ({size_mb:.1f} MB)")
else:
    # Maybe extracted directly to data/
    print("⚠️  train_30k_hard_data/ not found, checking data/ root...")
    for item in Path("data").iterdir():
        if item.is_dir() and item.name != "checkpoints":
            print(f"  📂 {item.name}/")

## 3. Create Config với Paths Đúng

In [ ]:
import yaml

# Load base config
with open("configs/stage1_30k_kaggle_t4.yaml", 'r') as f:
    config = yaml.safe_load(f)

# TODO: ADJUST THESE PATHS theo cấu trúc thực tế sau khi extract!
# Chạy cell trên trước để xem extracted structure

# Option 1: Nếu extract vào data/train_30k_hard_data/
config['data']['manifest'] = 'data/train_30k_hard_data/train_30k_hard.jsonl'
config['data']['image_root'] = 'data/train_30k_hard_data/images/'  # hoặc train_webp/
config['data']['vitpose_json'] = 'data/train_30k_hard_data/train_30k_hard_vitpose.json'

# Option 2: Nếu extract trực tiếp vào data/
# config['data']['manifest'] = 'data/train_30k_hard.jsonl'
# config['data']['image_root'] = 'data/images/'
# config['data']['vitpose_json'] = 'data/train_30k_hard_vitpose.json'

config['data']['boxes_json'] = 'data/boxes_30k.json'
config['model']['checkpoint'] = 'data/checkpoints/best.pth'

# Adjust batch size cho T4
config['train']['batch_size'] = 16  # Giảm xuống 12 nếu OOM
config['train']['num_workers'] = 2

# Save updated config
config_path = 'configs/stage1_kaggle_runtime.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✓ Config saved to: {config_path}")
print("\n📋 Key paths:")
print(f"  Manifest: {config['data']['manifest']}")
print(f"  Images: {config['data']['image_root']}")
print(f"  VitPose: {config['data']['vitpose_json']}")
print(f"  Boxes: {config['data']['boxes_json']}")
print(f"  Checkpoint: {config['model']['checkpoint']}")

## 4. Verify Paths (QUAN TRỌNG!)

In [ ]:
# MUST RUN: Verify all paths exist
import yaml
from pathlib import Path

with open('configs/stage1_kaggle_runtime.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

print("🔍 Verifying paths...\n")

paths_to_check = {
    'Manifest': cfg['data']['manifest'],
    'VitPose JSON': cfg['data']['vitpose_json'],
    'Boxes JSON': cfg['data']['boxes_json'],
    'Checkpoint': cfg['model']['checkpoint'],
    'Image root': cfg['data']['image_root'],
}

all_ok = True
for name, path in paths_to_check.items():
    p = Path(path)
    exists = p.exists()
    status = "✓" if exists else "❌"
    
    if exists:
        if p.is_dir():
            num_files = len(list(p.glob("*")))
            print(f"{status} {name}: {path} ({num_files} files)")
        else:
            size_mb = p.stat().st_size / (1024**2)
            print(f"{status} {name}: {path} ({size_mb:.1f} MB)")
    else:
        print(f"{status} {name}: {path} [NOT FOUND!]")
        all_ok = False

if all_ok:
    print("\n✅ All paths verified! Ready to train.")
else:
    print("\n❌ Some paths are missing! Fix paths in cell above before continuing.")
    raise FileNotFoundError("Missing required files!")

## 5. Sanity Check (MUST RUN!)

In [ ]:
# Test pipeline: Loss should drop từ ~2.0 → ~0.01 in 200 steps
print("🧪 Running sanity check (overfit one batch)...")
print("   Expected: Loss giảm nhanh sau 200 steps")
print("   Time: ~5-10 phút\n")

!python scripts/train.py \
    --config configs/stage1_kaggle_runtime.yaml \
    --init-from data/checkpoints/best.pth \
    --overfit-one-batch

print("\n✅ Sanity check passed! Pipeline is working.")

## 6. FULL TRAINING! 🔥

In [ ]:
# Full training với safety net
print("🚀 Starting full training...")
print("   Target: mAP 80% → 82-86%")
print("   Time: ~2-3 giờ on T4")
print("   Safety: Auto revert nếu mAP < 80%\n")

!python scripts/train.py \
    --config configs/stage1_kaggle_runtime.yaml \
    --init-from data/checkpoints/best.pth \
    --max-hours 11.5

print("\n🎉 Training completed!")

## 7. Evaluate Results

In [ ]:
import torch
from pathlib import Path

best_ckpt_path = Path("outputs/stage1_30k_t4/best.pth")

if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location='cpu')
    report = ckpt.get('report', {})
    
    print("📊 FINAL RESULTS:\n")
    print(f"  Epoch: {ckpt.get('epoch', 'N/A')}")
    print(f"  mAP: {report.get('mAP', 0.0)*100:.2f}%")
    print(f"  R@1: {report.get('R@1', 0.0)*100:.2f}%")
    print(f"  R@5: {report.get('R@5', 0.0)*100:.2f}%")
    print(f"  R@10: {report.get('R@10', 0.0)*100:.2f}%")
    print(f"  MRR: {report.get('MRR', 0.0)*100:.2f}%")
    
    # Success check
    baseline_map = 0.80
    final_map = report.get('mAP', 0.0)
    
    print("\n" + "="*50)
    if final_map >= 0.82:
        gain = (final_map - baseline_map) * 100
        print(f"✅ SUCCESS! mAP improved by +{gain:.2f}%")
        print(f"   From {baseline_map*100:.2f}% → {final_map*100:.2f}%")
    elif final_map >= baseline_map:
        gain = (final_map - baseline_map) * 100
        print(f"⚠️  MARGINAL: mAP improved by +{gain:.2f}%")
        print(f"   Target was +2%, got +{gain:.2f}%")
    else:
        loss = (baseline_map - final_map) * 100
        print(f"❌ FAILED: mAP dropped by -{loss:.2f}%")
        print(f"   Safety net should have reverted")
    print("="*50)
else:
    print("⚠️ Best checkpoint not found!")
    print("   Check outputs/stage1_30k_t4/ for logs")

## 8. Save Checkpoint for Download

In [ ]:
import shutil

output_dir = Path("/kaggle/working")
best_ckpt = Path("outputs/stage1_30k_t4/best.pth")

if best_ckpt.exists():
    shutil.copy(best_ckpt, output_dir / "stage1_best.pth")
    size_mb = best_ckpt.stat().st_size / (1024**2)
    print(f"✓ Checkpoint saved to {output_dir / 'stage1_best.pth'}")
    print(f"  Size: {size_mb:.1f} MB")
    print("\n📥 Download from Output tab (right side)")
else:
    print("⚠️ No checkpoint to save!")

## 9. Training Logs

In [ ]:
# Show last 50 lines of training log
log_file = Path("outputs/stage1_30k_t4/train.log")
if log_file.exists():
    !tail -50 {log_file}
else:
    print("⚠️ Log file not found")